In [13]:
import pandas as pd
import re
import nltk

NLTK_AVAILABLE = {}
for resource, resource_path in {
    "punkt": "tokenizers/punkt",
    "averaged_perceptron_tagger": "taggers/averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
    "wordnet": "corpora/wordnet",
    "stopwords": "corpora/stopwords",
}.items():
    try:
        nltk.data.find(resource_path)
        NLTK_AVAILABLE[resource] = True
    except LookupError:
        NLTK_AVAILABLE[resource] = False

print("NLTK resource status:", NLTK_AVAILABLE)

NLTK resource status: {'punkt': True, 'averaged_perceptron_tagger': True, 'averaged_perceptron_tagger_eng': False, 'wordnet': False, 'stopwords': True}


In [14]:
pain_points = '../output/challenges.csv'
expectations = '../output/expectations.csv'

df_pain_points = pd.read_csv(pain_points)
df_expectations = pd.read_csv(expectations)

In [15]:
df_pain_points.shape, df_expectations.shape

((424, 2), (433, 2))

In [16]:
# remove records with less than or equal to 4 words
df_pain_points = df_pain_points[df_pain_points['pain_points'].str.split().str.len() > 4]
df_expectations = df_expectations[df_expectations['expectations'].str.split().str.len() > 4]

In [17]:
df_pain_points.shape, df_expectations.shape

((324, 2), (309, 2))

In [18]:
df_pain_points.sample(5)

,focus_group,pain_points
360,DOCE Supervisors,Volume Challenges: IPS becomes cumbersome as c...
202,DOCE CommercialPermitElectrical Inspectors,Manual Scheduling Requirements: Inspectors oft...
325,DOCE Office Manager,Performance decreases when handling multiple r...
238,DOCE Fire Prevention Bureau,Multiple Search Locations: Information exists ...
9,Assessment Assessment Supervisors,Seasonal Review Process: Permit monitoring and...


In [19]:
from nltk import pos_tag, word_tokenize

def safe_word_tokenize(text):
    s = str(text)
    try:
        return word_tokenize(s)
    except LookupError:
        return re.findall(r"[A-Za-z]+", s)

def safe_pos_tag(tokens):
    try:
        return pos_tag(tokens)
    except LookupError:
        return [(tok, "NN") for tok in tokens]

def extract_keywords(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    keywords = [
        word.lower()
        for word, tag in tagged
        if tag.startswith("NN") or tag.startswith("JJ")
    ]
    return keywords

In [20]:
from nltk import RegexpParser

grammar = r"""
NP:
    {<JJ>*<NN.*>+}
"""

chunker = RegexpParser(grammar)

df_pain_points["processed_text"] = (
    df_pain_points["pain_points"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^A-Za-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

df_expectations["processed_text"] = (
    df_expectations["expectations"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^A-Za-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

def extract_phrases(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    tree = chunker.parse(tagged)

    phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == "NP":
            phrase = " ".join(
                word.lower()
                for word, _ in subtree.leaves()
            )
            phrases.append(phrase)

    return phrases

df_pain_points[["pain_points", "processed_text"]].head(3)

,pain_points,processed_text
1,Historical Record Management: Historical permi...,historical record management historical permit...
2,Incomplete Property History: IPS does not alwa...,incomplete property history ips does not alway...
3,Large Plan Viewing: Large plans and surveys ca...,large plan viewing large plans and surveys can...


In [21]:
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

tokens = []
for text in df_pain_points["processed_text"]:
    tokens.extend(safe_word_tokenize(text))

finder = BigramCollocationFinder.from_words(tokens)
finder.apply_freq_filter(3)

bigrams = finder.score_ngrams(
    BigramAssocMeasures().pmi
)

for bg, score in bigrams[:50]:
    print(bg, score)

('compliance', 'engine') 9.921097088106793
('making', 'it') 9.336134587385637
('department', 'dependencies') 9.113742166049189
('follow', 'up') 9.113742166049189
('single', 'source') 8.921097088106793
('paper', 'files') 8.698704666770345
('cross', 'department') 8.528779665328031
('city', 'departments') 8.2206573699657
('code', 'enforcement') 8.099095390084788
('too', 'many') 8.014206492498275
('time', 'consuming') 7.921097088106793
('a', 'single') 7.75117208666448
('switch', 'between') 7.59916899321943
('time', 'responding') 7.506059588827949
('zoning', 'reviews') 7.488137680830686
('heavily', 'on') 7.461665469469494
('spend', 'significant') 7.41859674757761
('data', 'entry') 7.376776571882981
('of', 'truth') 7.3361345873856365
('source', 'of') 7.3361345873856365
('requires', 'extensive') 7.2581320753843634
('supporting', 'documentation') 7.20685157044067
('other', 'tools') 7.113742166049188
('the', 'same') 7.113742166049188
('depends', 'on') 7.046627970190651
('processing', 'large') 7

In [22]:
from nltk.collocations import TrigramCollocationFinder
from nltk.metrics import TrigramAssocMeasures

finder = TrigramCollocationFinder.from_words(tokens)

finder.apply_freq_filter(3)

trigrams = finder.score_ngrams(
    TrigramAssocMeasures().pmi
)

In [23]:
from collections import Counter

phrase_counter = Counter()

for text in df_pain_points["processed_text"]:

    phrases = extract_phrases(text)

    phrase_counter.update(phrases)

raw_categories = (
    phrase_counter
    .most_common(100)
)

print(raw_categories)

[('historical record management historical permits and property records often require searching ips together with onbase because historical information is not consolidated', 1), ('incomplete property history ips does not always provide a complete historical picture for permits certificates of occupancy or property changes', 1), ('large plan viewing large plans and surveys can not be viewed effectively within ips', 1), ('limited historical visibility historical records are frequently needed to determine legal occupancy and previous property conditions but are not always easily accessible', 1), ('historical property research research often requires reviewing permits surveys engineering records and historical documents together', 1), ('resubdivision coordination parcel updates require coordination with engineering and county mapping', 1), ('permit driven workload most assessment work begins with permit activity although unpermitted work may still be discovered', 1), ('seasonal review proc

In [24]:
df_pain_points.head(3)

,focus_group,pain_points,processed_text
1,Assessment Assessment Supervisors,Historical Record Management: Historical permi...,historical record management historical permit...
2,Assessment Assessment Supervisors,Incomplete Property History: IPS does not alwa...,incomplete property history ips does not alway...
3,Assessment Assessment Supervisors,Large Plan Viewing: Large plans and surveys ca...,large plan viewing large plans and surveys can...


In [29]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model = "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

pain_point_preds = classifier(df_pain_points["processed_text"].tolist())
expectation_preds = classifier(df_expectations["processed_text"].tolist())

df_pain_points["sentiment"] = [pred["label"].lower() for pred in pain_point_preds]
df_expectations["sentiment"] = [pred["label"].lower() for pred in expectation_preds]

df_pain_points[["processed_text", "sentiment"]].head(3)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,processed_text,sentiment
1,historical record management historical permit...,neutral
2,incomplete property history ips does not alway...,neutral
3,large plan viewing large plans and surveys can...,neutral


In [30]:
# total pain points per focus group (show all groups)
pain_points_summary = df_pain_points.groupby("focus_group", dropna=False).agg(
    total_pain_points=pd.NamedAgg(column="pain_points", aggfunc="count"),
    negative_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_pain_points["focus_group"].nunique(dropna=False))
print("rows in summary:", len(pain_points_summary))

pain_points_summary.sort_values(by=["total_pain_points", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16
rows in summary: 16


,focus_group,total_pain_points,negative_pain_points,positive_pain_points,neutral_pain_points
6,DOCE Admin Aide,34,14,0,20
8,DOCE CommercialPermitElectrical Inspectors,30,14,0,16
10,DOCE Housing Inspectors,30,11,0,19
7,DOCE Building Inspectors,29,18,0,11
11,DOCE Office Manager,28,9,0,19
12,DOCE Supervisors,26,10,0,16
9,DOCE Fire Prevention Bureau,25,12,0,13
4,CPO CPO Co-Ordinator,22,8,0,14
5,CPO Central Permit Office,21,7,0,14
15,NBD NBD Internal,14,4,0,10


In [32]:
# (in case any positive sentiment is detected) convert positive sentiment to neutral for pain points
# df_pain_points.loc[df_pain_points['sentiment'] == 'positive', 'sentiment'] = 'neutral'

In [38]:
# total expectations per focus group (show all groups)
expectations_summary = df_expectations.groupby("focus_group", dropna=False).agg(
    total_expectations=pd.NamedAgg(column="expectations", aggfunc="count"),
    negative_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_expectations["focus_group"].nunique(dropna=False))
print("rows in summary:", len(expectations_summary))

expectations_summary.sort_values(by=["total_expectations", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16
rows in summary: 16


,focus_group,total_expectations,negative_expectations,positive_expectations,neutral_expectations
10,DOCE Housing Inspectors,31,4,17,10
12,DOCE Supervisors,30,4,19,7
7,DOCE Building Inspectors,27,4,13,10
5,CPO Central Permit Office,26,4,18,4
6,DOCE Admin Aide,26,3,17,6
8,DOCE CommercialPermitElectrical Inspectors,26,1,14,11
9,DOCE Fire Prevention Bureau,26,7,12,7
4,CPO CPO Co-Ordinator,25,0,18,7
11,DOCE Office Manager,21,0,15,6
3,CPC CPC,13,5,6,2


In [39]:
# display expectations that are negative
neg_expectations = df_expectations.loc[
    df_expectations["sentiment"] == "negative",
    ["focus_group", "expectations", "sentiment"]
]
neg_expectations.sample(n=min(10, len(neg_expectations)), random_state=42)

,focus_group,expectations,sentiment
358,DOCE Supervisors,"Single Parcel View: Consolidate permits, viola...",negative
409,LAW Law,Property-Level Case View: Provide consolidated...,negative
34,CPC CPC,Separated Violation Records: Clearly distingui...,negative
128,DOCE Admin Aide,Auto-populate commonly used complaint and insp...,negative
270,DOCE Housing Inspectors,Integrated Lead Information: Display lead-risk...,negative
258,DOCE Fire Prevention Bureau,Comprehensive Fire Prevention Platform: Suppor...,negative
37,CPC CPC,Reduced Duplicate Data Entry: Eliminate repeti...,negative
259,DOCE Fire Prevention Bureau,Real-Time Field Operations: Enable inspectors ...,negative
241,DOCE Fire Prevention Bureau,Integrated Fire Operations: Connect IPS violat...,negative
133,DOCE Admin Aide,Reduce crashes and memory-related errors.,negative


In [40]:
#convert negative sentiment to neutral for expectations
df_expectations.loc[df_expectations['sentiment'] == 'negative', 'sentiment'] = 'neutral'

In [41]:
#save as csv
df_pain_points.to_csv("../output/clean_challenges.csv", index=False)
df_expectations.to_csv("../output/clean_expectations.csv", index=False)